In [49]:
import pandas as pd

In [50]:
input_file = "../data/DataStar.xlsx"
output_file = "../output/DataStar_Schema_Output.xlsx"

In [51]:
# Reading source data from excel
df = pd.read_excel(input_file,sheet_name='STAR DATA')
df = df.dropna(how = "all")
df.head()

,Date,Store Location,Food Department,Total Value (000's)
1,1st Sep 1994,Wolverhampton,Meat,234.0
2,2nd Sep 1994,Grimsby,Baked,143.0
3,3rd Sep 1994,Durham,Confectionary,482.0
4,4th Sep 1994,Swansea,Floral,471.0
5,5th Sep 1994,Wolverhampton,Dairy,883.0


In [52]:
df.columns = df.columns.str.strip()

In [53]:
# Creat Date Dimension
dim_date = df[["Date"]].drop_duplicates().reset_index(drop= True)
dim_date.insert(0, "DateKey", range(101,101 + len(dim_date)))

In [54]:
# Creat Store Dimension
dim_store = df[["Store Location"]].drop_duplicates().reset_index(drop= True)
dim_store.insert(0, "StoreKey", range(201,201 + len(dim_store)))

In [55]:
# Creat Food Deparment Dimension
dim_deparment = df[["Food Department"]].drop_duplicates().reset_index(drop= True)
dim_deparment.insert(0, "DepartmentKey", range(301,301 + len(dim_deparment)))

In [56]:
# Merge keys into fact table 
fact_sales = df.merge(dim_date, on="Date", how="left")
fact_sales = fact_sales.merge(dim_store, on="Store Location", how="left")
fact_sales = fact_sales.merge(dim_deparment, on="Food Department", how="left")

In [57]:
# Keep only fact table fields
fact_sales = fact_sales[
    ["DateKey", "StoreKey", "DepartmentKey", "Total Value (000's)"]
    ]

In [58]:
# Remove "SalesFactKey" if it already exists
if "SalesFactKey" in fact_sales.columns: 
    fact_sales = fact_sales.drop(columns=["SalesFactKey"])

# Inser "SalesFactKey" safely
fact_sales.insert(0, "SalesFactKey", range(1, 1 + len(fact_sales)))

print ("Data Warehouse Schema has been created successfully!")

Data Warehouse Schema has been created successfully!


In [59]:
# Write STAR schemas to EXCEL
with pd.ExcelWriter(output_file, "openpyxl") as writer:
    df.to_excel(writer, sheet_name="Original_Data", index=False)
    dim_date.to_excel(writer, sheet_name="Dim_Date", index=False)
    dim_store.to_excel(writer, sheet_name="Dim_Store", index=False)
    dim_deparment.to_excel(writer, sheet_name="Dim_Department", index=False)
    fact_sales.to_excel(writer, sheet_name="Fact_Sales", index=False)

print ("Data Warehouse Schema has been exported to Excel successfully!", output_file)

Data Warehouse Schema has been exported to Excel successfully! ../output/DataStar_Schema_Output.xlsx
